# COMFORT Eval — Test04 Analysis

Single-image MCQ on visual-mark scenes. 4-class relation prediction from the person's perspective.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

RELATIONS = ["front", "behind", "left", "right"]

RELATION_MAP = {
    "infrontof": "front",
    "totheleft": "left",
    "totheright": "right",
    "behind": "behind",
}

TRUE_LABEL_GROUPS = {
    "front_behind": ["front", "behind"],
    "left_right":   ["left",  "right"],
}

OPPOSITE_RELATION = {"front": "behind", "behind": "front", "left": "right", "right": "left"}

# Test02: correct = always "front" (cam_pov_front).
# When wrong, model picked the distractor whose direction is relation-dependent:
#   front/behind -> cam_pov_back  -> "behind"
#   left         -> cam_pov_left  -> "left"
#   right        -> cam_pov_right -> "right"
TEST02_DISTRACTOR_DIRECTION = {
    "front":  "behind",
    "behind": "behind",
    "left":   "left",
    "right":  "right",
}

# ── Shared helpers ─────────────────────────────────────────────────────────

def extract_camera_perspective_comfort(image_path):
    image_path = str(image_path)
    parts = os.path.dirname(image_path).split("/")
    true_relation = parts[-2]
    folder_name = parts[-1]
    segments = folder_name.split("__")
    cam_segment = segments[-1]
    camera_position = cam_segment.split("_")[-1]

    camera_perspective = None
    if (true_relation == "behind" and camera_position == "left") or (true_relation == "infrontof" and camera_position == "right") or (true_relation == "totheleft" and camera_position == "front") or (true_relation == "totheright" and camera_position == "back"):
        camera_perspective = "right"
    elif (true_relation == "behind" and camera_position == "right") or (true_relation == "infrontof" and camera_position == "left") or (true_relation == "totheleft" and camera_position == "back") or (true_relation == "totheright" and camera_position == "front"):
        camera_perspective = "left"
    elif (true_relation == "behind" and camera_position == "front") or (true_relation == "infrontof" and camera_position == "back") or (true_relation == "totheleft" and camera_position == "right") or (true_relation == "totheright" and camera_position == "left"):
        camera_perspective = "behind"
    elif (true_relation == "behind" and camera_position == "back") or (true_relation == "infrontof" and camera_position == "front") or (true_relation == "totheleft" and camera_position == "left") or (true_relation == "totheright" and camera_position == "right"):
        camera_perspective = "front"
    return true_relation, camera_perspective


def extract_letter_to_relation(prompt):
    prompt = str(prompt)
    mapping = {}
    for letter in ["A", "B", "C", "D"]:
        pattern = rf"{letter}\.\s*(.*?)(?=(?:A|B|C|D)\.\s|$)"
        m = re.search(pattern, prompt, flags=re.IGNORECASE | re.DOTALL)
        if not m:
            continue
        text = m.group(1).lower()
        if "front" in text:    mapping[letter] = "front"
        elif "behind" in text: mapping[letter] = "behind"
        elif "left" in text:   mapping[letter] = "left"
        elif "right" in text:  mapping[letter] = "right"
    return mapping


def plot_confusion_on_ax(df, ax, title, true_labels):
    plot_df = df[
        df["correct_relation"].isin(true_labels) &
        df["pred_relation"].isin(RELATIONS)
    ].copy()

    if len(plot_df) == 0:
        ax.axis("off")
        ax.set_title(f"{title}\n(no data)")
        return

    cm_df = pd.crosstab(
        plot_df["correct_relation"],
        plot_df["pred_relation"],
        normalize="index"
    ).reindex(index=true_labels, columns=RELATIONS, fill_value=0.0)

    im = ax.imshow(cm_df.values, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(RELATIONS)))
    ax.set_xticklabels(RELATIONS)
    ax.set_yticks(range(len(true_labels)))
    ax.set_yticklabels(true_labels)
    for i in range(cm_df.shape[0]):
        for j in range(cm_df.shape[1]):
            ax.text(j, i, f"{cm_df.iloc[i, j]:.2f}", ha="center", va="center")
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")


# ── Test01 loader ─────────────────────────────────────────────────────────

def load_and_prepare_pair(csv_path):
    df = pd.read_csv(csv_path)

    def get_pred_relation(row):
        return extract_letter_to_relation(row["mcq_prompt"]).get(str(row["pred_letter"]).strip(), None)

    df["pred_relation"] = df.apply(get_pred_relation, axis=1)
    df["correct_relation"] = df["correct_relation"].astype(str).str.strip().str.lower()

    r1 = df["image_path_1"].apply(extract_camera_perspective_comfort)
    r2 = df["image_path_2"].apply(extract_camera_perspective_comfort)
    df["object_position"]   = r1.apply(lambda x: RELATION_MAP.get(x[0], x[0]))
    df["cam_perspective_1"] = r1.apply(lambda x: x[1])
    df["cam_perspective_2"] = r2.apply(lambda x: x[1])

    df["cam_view_1"] = df["cam_view_1"].astype(str).str.strip()
    df["cam_view_2"] = df["cam_view_2"].astype(str).str.strip()
    df["view2_short"] = df["cam_view_2"].str.replace("cam_", "", regex=False)

    df["is_correct"]  = df["pred_letter"].astype(str).str.strip() == df["correct_letter"].astype(str).str.strip()
    df["is_opposite"] = df["pred_letter"].astype(str).str.strip() == df["opposite_letter"].astype(str).str.strip()
    return df


# ── Test02 loader ─────────────────────────────────────────────────────────

def load_and_prepare_pov(csv_path):
    df = pd.read_csv(csv_path)
    df["correct_relation"] = df["correct_relation"].astype(str).str.strip().str.lower()
    df["correct_relation"] = df["correct_relation"].map(RELATION_MAP).fillna(df["correct_relation"])
    df["is_correct"] = df["correct"].astype(str).str.lower().isin(["true", "1"])
    df["cam_view_external_short"] = df["cam_view_external"].str.replace("cam_", "", regex=False)
    return df


print("Helpers loaded.")

---
## Test 04 — Single-Image Visual-Marks MCQ

Input: 1 visual-mark image from `comfort_human_car_visual_marks`.  
Task: 4-choice relation prediction (`front` / `behind` / `left` / `right`) from the person's perspective.  
Random baseline = 0.25.


In [ ]:
TEST04_DIR = "./results"
models04   = ["qwen3vl", "qwen3_5vl"]
lengths04  = ["short", "middle", "long"]


In [ ]:
def extract_cam_view_from_path(image_path):
    image_path = str(image_path)
    m = re.search(r"__cam_(back|front|left|right)", image_path)
    return m.group(1) if m else None


def load_and_prepare_visual_marks(csv_path):
    df = pd.read_csv(csv_path)

    def get_pred_relation(row):
        letter = str(row["pred_letter"]).strip().upper()
        return extract_letter_to_relation(row["mcq_prompt"]).get(letter, None)

    df["correct_relation"] = df["correct_relation"].astype(str).str.strip().str.lower()
    df["correct_relation"] = df["correct_relation"].map(RELATION_MAP).fillna(df["correct_relation"])
    df["opposite_relation"] = df["opposite_relation"].astype(str).str.strip().str.lower()
    df["opposite_relation"] = df["opposite_relation"].map(RELATION_MAP).fillna(df["opposite_relation"])
    df["pred_letter"] = df["pred_letter"].astype(str).str.strip().str.upper()
    df["correct_letter"] = df["correct_letter"].astype(str).str.strip().str.upper()
    df["opposite_letter"] = df["opposite_letter"].astype(str).str.strip().str.upper()
    df["pred_relation"] = df.apply(get_pred_relation, axis=1)
    df["is_correct"] = df["pred_letter"] == df["correct_letter"]
    df["is_opposite"] = df["pred_letter"] == df["opposite_letter"]
    df["cam_view_short"] = df["image_path"].apply(extract_cam_view_from_path)
    return df


all_dfs_04 = defaultdict(dict)

for model in models04:
    print(f"\n===== MODEL: {model} =====")
    for length in lengths04:
        csv_path = os.path.join(TEST04_DIR, f"mcq_{length}_{model}.csv")
        if not os.path.exists(csv_path):
            print(f"  Missing: {csv_path}")
            continue
        df = load_and_prepare_visual_marks(csv_path)
        all_dfs_04[model][length] = df
        print(
            f"  Loaded {length}: {len(df)} rows  |  "
            f"overall acc = {df['is_correct'].mean():.3f}  |  "
            f"opposite rate = {df['is_opposite'].mean():.3f}"
        )


### 1 — Overall accuracy by model and prompt length

In [ ]:
rows = []
for model in models04:
    for length in lengths04:
        if length not in all_dfs_04.get(model, {}):
            continue
        df = all_dfs_04[model][length]
        rows.append({
            "model": model,
            "length": length,
            "n": len(df),
            "accuracy": df["is_correct"].mean(),
            "opposite_rate": df["is_opposite"].mean(),
            "invalid_pred_rate": df["pred_relation"].isna().mean(),
        })

summary04 = pd.DataFrame(rows)
display(summary04.assign(
    accuracy=lambda x: x["accuracy"].round(3),
    opposite_rate=lambda x: x["opposite_rate"].round(3),
    invalid_pred_rate=lambda x: x["invalid_pred_rate"].round(3),
))

if len(summary04) > 0:
    x = np.arange(len(lengths04))
    width = 0.35
    fig, ax = plt.subplots(figsize=(7, 4))

    for k, model in enumerate(models04):
        sub = summary04[summary04["model"] == model].set_index("length")
        vals = [sub.loc[length, "accuracy"] if length in sub.index else np.nan for length in lengths04]
        bars = ax.bar(x + (k - (len(models04)-1)/2) * width, vals, width, label=model)
        for bar, val in zip(bars, vals):
            if not np.isnan(val):
                ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                        f"{val:.2f}", ha="center", va="bottom", fontsize=8)

    ax.axhline(0.25, color="gray", linestyle="--", linewidth=0.8, label="random (0.25)")
    ax.set_xticks(x)
    ax.set_xticklabels(lengths04)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Accuracy")
    ax.set_title("Test04: overall accuracy by model and prompt length")
    ax.legend()
    plt.tight_layout()
    plt.show()


### 2 — Confusion matrices (short / middle / long)

In [ ]:
for model in models04:
    if not all_dfs_04.get(model):
        continue

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for i, length in enumerate(lengths04):
        if length not in all_dfs_04[model]:
            axes[i].axis("off")
            continue
        plot_confusion_on_ax(
            all_dfs_04[model][length], axes[i],
            title=f"{length} (n={len(all_dfs_04[model][length])})",
            true_labels=RELATIONS,
        )
    fig.suptitle(f"{model} — Test04: single-image visual-marks MCQ", fontsize=14)
    plt.tight_layout()
    plt.show()


### 2b — Confusion matrices grouped by camera-perspective object position

Rows are grouped by `object_position_under_camera_perspective`: where the object appears from the camera/viewer perspective.  
Each confusion matrix still uses the true person-perspective relation as the label (`correct_relation`).


In [ ]:
EXAMPLE_IMAGES_04 = {
    "left":   "/home/zzou/COMFORT/data/comfort_human_car_visual_marks/behind/basketball__behind__cam_right/0.png",
    "right":  "/home/zzou/COMFORT/data/comfort_human_car_visual_marks/behind/basketball__behind__cam_left/0.png",
    "front":  "/home/zzou/COMFORT/data/comfort_human_car_visual_marks/behind/basketball__behind__cam_back/0.png",
    "behind": "/home/zzou/COMFORT/data/comfort_human_car_visual_marks/behind/basketball__behind__cam_front/0.png",
}

OBJECT_POSITIONS_UNDER_CAMERA = ["left", "right", "front", "behind"]

for model in models04:
    if not all_dfs_04.get(model):
        continue

    dfs = {}
    for length in lengths04:
        if length not in all_dfs_04[model]:
            continue
        df = all_dfs_04[model][length].copy()
        parsed = df["image_path"].apply(extract_camera_perspective_comfort)
        df["object_position"] = parsed.apply(lambda x: RELATION_MAP.get(x[0], x[0]))
        df["object_position_under_camera_perspective"] = parsed.apply(lambda x: x[1])
        dfs[length] = df
        all_dfs_04[model][length] = df

    fig, axes = plt.subplots(4, 4, figsize=(20, 16))

    for j, obj_pos in enumerate(OBJECT_POSITIONS_UNDER_CAMERA):
        img_path = EXAMPLE_IMAGES_04[obj_pos]
        if os.path.exists(img_path):
            axes[j][0].imshow(plt.imread(img_path))
            axes[j][0].axis("off")
            axes[j][0].set_title(
                f"obj_pos_under_camera_perspective={obj_pos}",
                fontsize=9,
            )
        else:
            axes[j][0].axis("off")
            axes[j][0].set_title(f"Missing example: {obj_pos}", fontsize=9)

        for i, length in enumerate(lengths04):
            ax = axes[j][i + 1]
            if length not in dfs:
                ax.axis("off")
                continue
            sub = dfs[length][
                dfs[length]["object_position_under_camera_perspective"] == obj_pos
            ]
            plot_confusion_on_ax(
                sub,
                ax,
                title=f"{length} (n={len(sub)})",
                true_labels=RELATIONS,
            )

    fig.suptitle(
        f"{model} — Test04: object position under camera perspective (rows) × length (cols)",
        fontsize=14,
    )
    plt.tight_layout()
    plt.show()


### 3 — Accuracy heatmap: relation × camera view

In [ ]:
VIEW_SHORT = ["back", "front", "left", "right"]

for model in models04:
    if not all_dfs_04.get(model):
        continue

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for i, length in enumerate(lengths04):
        if length not in all_dfs_04[model]:
            axes[i].axis("off")
            continue

        df = all_dfs_04[model][length]
        mat = np.full((len(RELATIONS), len(VIEW_SHORT)), np.nan)
        annot = []
        for r, rel in enumerate(RELATIONS):
            row_ann = []
            for c, view in enumerate(VIEW_SHORT):
                sub = df[(df["correct_relation"] == rel) & (df["cam_view_short"] == view)]
                if len(sub) > 0:
                    mat[r, c] = sub["is_correct"].mean()
                    row_ann.append(f"{mat[r,c]:.2f}\n(n={len(sub)})")
                else:
                    row_ann.append("")
            annot.append(row_ann)

        sns.heatmap(
            mat, ax=axes[i], annot=annot, fmt="",
            cmap="RdYlGn", vmin=0, vmax=1, linewidths=0.5,
            xticklabels=VIEW_SHORT, yticklabels=RELATIONS,
            annot_kws={"size": 8}, cbar=(i == 2),
        )
        axes[i].set_title(length)
        axes[i].set_xlabel("Camera view")
        axes[i].set_ylabel("Correct relation" if i == 0 else "")

    fig.suptitle(f"{model} — Test04: accuracy by relation × camera view", fontsize=14)
    plt.tight_layout()
    plt.show()


### 4 — Per-relation and per-camera tables

In [ ]:
for model in models04:
    if not all_dfs_04.get(model):
        continue

    print(f"\n===== {model}: per-relation accuracy =====")
    rows = []
    for length in lengths04:
        if length not in all_dfs_04[model]:
            continue
        df = all_dfs_04[model][length]
        for rel in RELATIONS:
            sub = df[df["correct_relation"] == rel]
            rows.append({
                "length": length,
                "relation": rel,
                "n": len(sub),
                "accuracy": sub["is_correct"].mean() if len(sub) > 0 else np.nan,
            })
    rel_tbl = pd.DataFrame(rows).pivot(index="relation", columns="length", values="accuracy")
    display(rel_tbl.round(3))

    print(f"\n===== {model}: per-camera accuracy =====")
    rows = []
    for length in lengths04:
        if length not in all_dfs_04[model]:
            continue
        df = all_dfs_04[model][length]
        for view in VIEW_SHORT:
            sub = df[df["cam_view_short"] == view]
            rows.append({
                "length": length,
                "camera_view": view,
                "n": len(sub),
                "accuracy": sub["is_correct"].mean() if len(sub) > 0 else np.nan,
            })
    cam_tbl = pd.DataFrame(rows).pivot(index="camera_view", columns="length", values="accuracy")
    display(cam_tbl.round(3))


### 5 — Predicted relation distribution

In [ ]:
for model in models04:
    if not all_dfs_04.get(model):
        continue

    for length in lengths04:
        if length not in all_dfs_04[model]:
            continue
        df = all_dfs_04[model][length]
        pred_dist = (
            df["pred_relation"]
            .value_counts(normalize=True)
            .reindex(RELATIONS)
            .fillna(0)
            .round(3)
            .to_frame(name=f"{model}_{length}")
        )
        display(pred_dist)
